# Graph structures

A unique feature to Graph Neural Networks is that these models learn by the enforcement of a spatial structure; a graph. When applied to epidemiological problems, GNNs should have access to meaningful graph structures which encode the (spatial) relations between the different entities modelled. A graph consists of **nodes** (entities) connected by **edges** (representing relationships between them). Edges may be weighted to quantify connection strength. In epidemiological applications, nodes may represent geographic regions such as districts or municipalities, while edges encode relationships relevant to disease transmission—including travel patterns or spatial proximity. In this project, a graph structure $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ represents Germany, with $N = |\mathcal{V}|$ nodes representing the districts and edges $\mathcal{E}$ encoding various forms of spatial connectivity. GNNs employ message-passing mechanisms that enable each node to aggregate information from its connected neighbors, effectively capturing the relational dynamics within the network. This ability to integrate diverse types of information across graph structures makes GNNs particularly promising for epidemiological modeling (Kraemer, 2025)


### Graph Structures used
In this project, multiple graph construction strategies are explored, to capture different aspects of spatial connectivity. Per administrative unit in Germany, albeit *NUTS1*, *NUTS2* or *NUTS3*, we model epidemiological timeseries, representing incidence rates on casenumbers of the respective infectious disease. The following classes of graph structures are studied:

- **Identity graph**:  A baseline graph structure in which each node is only connected to itself. Therefore, for the prediction of the next state of node $i$, only information of its own past state is used.
- **Mesh graph**: A baseline graph structure in which each node is connected to all other nodes by equal weight.
- **Boolean Neighbors**: A graph representing geographical information by connecting every node to the nodes it shares a geographic border with.
- **Gravity-model**: Graphs encoding geographical and socio-demographic information based on the gravity model of spatial interaction. For any pair of nodes $i, j$, edge weights are computed analogously to the gravitational force between two objects. The connection strength is a function of the population size of $i, j$, and the inverse of their distance. Two variations are used, one with $k=3$ (gravity1) and one with $k=7$ (gravity2).
- **Commuter-based**: Graphs representing commuting data made available by the Bundesagentur für Arbeit \cite{Arbeitsagentur}. Two variations are used based on the commuting data for 2024, one with $k=3$ (commuter 1) and one in which each node's connections are kept, so long as there are more than 1 000 daily commuters between them (commuter2).

 In addition, a threshold of $k$ connections per node may be implemented, emphasizing strong connections over many connections.


In [1]:
from src.utils import get_data_env
from src.dataloading import EpiConfig, EpiDataOrchestrator
from src.graphconstruction import StaticGraphOrchestrator, DynamicGraphOrchestrator

disease_name    = 'influenza'
nuts_level      = 'nuts1'
min_date        = '2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False

horizon_size    = 1
horizon_leadtime= 3
sequence_length = 1
lag_num         = 1

config_nuts1 = EpiConfig(
    disease             = disease_name,
    min_date            = min_date, 
    max_date            = max_date,
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = 'nuts1',
    log_transform       = ['incidence'],
    split_berlin        = split_berlin,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'incidence',
    lag_column          = 'incidence',  
    verbose             = 0  
    )     
# 
config_nuts2 = EpiConfig(
    disease             = disease_name,
    min_date            = min_date, 
    max_date            = max_date,
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = 'nuts2',
    log_transform       = ['incidence'],
    split_berlin        = split_berlin,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'incidence',
    lag_column          = 'incidence',  
    verbose             = 0  
    )    



config_nuts3 = EpiConfig(
    disease             = disease_name,
    min_date            = min_date, 
    max_date            = max_date,
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = 'nuts3',
    log_transform       = ['incidence'],
    split_berlin        = split_berlin,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'incidence',
    lag_column          = 'incidence',  
    verbose             = 0  
    )      

data_orchestrator_nuts1 = EpiDataOrchestrator(config_nuts1).build()
data_orchestrator_nuts2 = EpiDataOrchestrator(config_nuts2).build()
data_orchestrator_nuts3 = EpiDataOrchestrator(config_nuts3).build()

# NUTS 1 graph structures

In [4]:
static_graphconstruction_nuts1 = StaticGraphOrchestrator(data_orchestrator_nuts1, id_col='nuts_node')
static_graphconstruction_nuts1.generate_graphstructure(method = 'identity', 
                                                       graphname='identity_graph', 
                                                       self_connection='max')

# Neighbors
#   boolean-self
static_graphconstruction_nuts1.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='mean', 
                                                       graphname = 'geographical_neighbors1')
#   boolean-nonself
static_graphconstruction_nuts1.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='0', 
                                                       graphname = 'geographical_neighbors2')
#   numerical-self
static_graphconstruction_nuts1.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection = 'mean', 
                                                       normalization='rowwise', 
                                                       graphname = 'geographical_neighbors3')
#   numerical-nonself
static_graphconstruction_nuts1.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='0', 
                                                       normalization='rowwise', 
                                                       graphname = 'geographical_neighbors4')

# static_graphconstruction_nuts1.graph_registry.save_graphentry('ALL')

normalization method is None
    ✓ Graph registered  : identity_graph successfully registered
normalization method is None
    ✓ Graph registered  : geographical_neighbors1 successfully registered
normalization method is None
    ✓ Graph registered  : geographical_neighbors2 successfully registered
    ✓ Graph registered  : geographical_neighbors3 successfully registered
    ✓ Graph registered  : geographical_neighbors4 successfully registered


# NUTS 2 graph structures

In [5]:
static_graphconstruction_nuts2 = StaticGraphOrchestrator(data_orchestrator_nuts2)
static_graphconstruction_nuts2.generate_graphstructure(method = 'identity', 
                                                       graphname='identity_graph', 
                                                       self_connection='max')

# Neighbors
#   boolean-self
static_graphconstruction_nuts2.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='mean', 
                                                       graphname = 'geographical_neighbors1')
#   boolean-nonself
static_graphconstruction_nuts2.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='0', 
                                                       graphname = 'geographical_neighbors2')
#   numerical-self
static_graphconstruction_nuts2.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection = 'mean', 
                                                       normalization='rowwise', 
                                                       graphname = 'geographical_neighbors3')
#   numerical-nonself
static_graphconstruction_nuts2.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='0', 
                                                       normalization='rowwise', 
                                                       graphname = 'geographical_neighbors4')

#   sparse - long distance gravity model
static_graphconstruction_nuts2.generate_graphstructure(method             = 'gravity_model', 
                                                 self_connection    = '0',
                                                 max_distance       = 1000_000,
                                                 top_k              = 3,
                                                 alpha              = 1,
                                                 decay              = 1,
                                                 normalization     = 'rowwise',
                                                 graphname = 'gravity1'
                                                 )
#   dense - long distance gravity model
static_graphconstruction_nuts2.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 10,
                                          alpha              = 1,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity2'
                                          )
#   sparse - short distance gravity model
static_graphconstruction_nuts2.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 3,
                                          alpha              = 2,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity3'
                                          )
#   dense - short distance gravity model
static_graphconstruction_nuts2.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 10,
                                          alpha              = 2,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity4'
                                          )
#   medium-dense - medium-distance gravity model
static_graphconstruction_nuts2.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 7,
                                          alpha              = 1.5,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity5'
                                          )

### no nuts2-level aggregation of commuting data done


static_graphconstruction_nuts2.graph_registry.save_graphentry('ALL')

normalization method is None
    ✓ Graph registered  : identity_graph successfully registered
normalization method is None
    ✓ Graph registered  : geographical_neighbors1 successfully registered
normalization method is None
    ✓ Graph registered  : geographical_neighbors2 successfully registered
    ✓ Graph registered  : geographical_neighbors3 successfully registered
    ✓ Graph registered  : geographical_neighbors4 successfully registered
    ✓ Graph registered  : gravity1 successfully registered
    ✓ Graph registered  : gravity2 successfully registered
    ✓ Graph registered  : gravity3 successfully registered
    ✓ Graph registered  : gravity4 successfully registered
    ✓ Graph registered  : gravity5 successfully registered
/local/job_23816311/wissdaten/ZKI-PH4/deschrijvers_wissdaten/project_utilities/infectious_disease_gnn/graphs/nuts2/identity_graph
    ✓ GraphEntry Saved  : identity_graph has been saved
/local/job_23816311/wissdaten/ZKI-PH4/deschrijvers_wissdaten/project_ut

# NUTS 3 - graph structures

#### claudes additions

In [10]:
static_graphconstruction_nuts3 = StaticGraphOrchestrator(data_orchestrator_nuts3, id_col='nuts_node')

# 1. Pure neighbors — no self-loop at all, no normalization
#    GCNConv will do its own normalisation. If rowwise + GCNConv was double-normalising,
#    this is the clean baseline that removes your normalisation entirely.
static_graphconstruction_nuts3.generate_graphstructure(
    method          = 'geographic_neighbors',
    self_connection = '0',          # no self-loop
    normalization   = None,         # let GCNConv handle it internally
    graphname       = 'diag_neighbors_raw'
)

# 2. Neighbors with self-loop, no normalisation
#    Adds self-loop so GCNConv gets it as part of its own D^{-1/2} A D^{-1/2}.
#    This is how GCNConv is *designed* to be used.
static_graphconstruction_nuts3.generate_graphstructure(
    method          = 'geographic_neighbors',
    self_connection = 'mean',
    normalization   = None,
    graphname       = 'diag_neighbors_selfloop_raw'
)

# 3. Identity, no normalisation
#    Apples-to-apples comparison with diag_neighbors_raw.
#    Your existing identity graph uses self_connection='max', which inflates weights.
static_graphconstruction_nuts3.generate_graphstructure(
    method          = 'identity',
    self_connection = 'max',
    normalization   = None,         # remove the rowwise that was fighting GCNConv
    graphname       = 'diag_identity_raw'
)

static_graphconstruction_nuts3.generate_graphstructure(
    method          = 'gravity_model',
    graphname       = 'gravity_decay2_selfloop',
    decay           = 2.2,
    max_distance    = 300_000,
    self_connection = 'mean',   # self-loop weight = mean of node's edge weights
    normalization   = 'rowwise',
)

static_graphconstruction_nuts3.graph_registry.rename_entry('diag_neighbors_raw', 'test1')
static_graphconstruction_nuts3.graph_registry.rename_entry('diag_neighbors_selfloop_raw', 'test2')
static_graphconstruction_nuts3.graph_registry.rename_entry('diag_identity_raw', 'test3')
static_graphconstruction_nuts3.graph_registry.rename_entry('gravity_decay2_selfloop', 'test4')

static_graphconstruction_nuts3.graph_registry.save_graphentry(['test4'])

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


normalization method is None
    ✓ Graph registered  : diag_neighbors_raw successfully registered
normalization method is None
    ✓ Graph registered  : diag_neighbors_selfloop_raw successfully registered
normalization method is None
    ✓ Graph registered  : diag_identity_raw successfully registered
    ✓ Graph registered  : gravity_decay2_selfloop successfully registered
    ✓ Graph registered  : test1 successfully registered
    ✓ Graph removed     : diag_neighbors_raw has been deregistered
    ✓ Graph registered  : test2 successfully registered
    ✓ Graph removed     : diag_neighbors_selfloop_raw has been deregistered
    ✓ Graph registered  : test3 successfully registered
    ✓ Graph removed     : diag_identity_raw has been deregistered
    ✓ Graph registered  : test4 successfully registered
    ✓ Graph removed     : gravity_decay2_selfloop has been deregistered
/local/job_24503780/wissdaten/ZKI-PH4/deschrijvers_wissdaten/project_utilities/infectious_disease_gnn/graphs/nuts3/test

In [5]:
static_graphconstruction_nuts3 = StaticGraphOrchestrator(data_orchestrator_nuts3, id_col='nuts_node')
static_graphconstruction_nuts3.generate_graphstructure(method = 'identity', 
                                                       graphname='identity_graph', 
                                                       self_connection='max')

# Neighbors
#   boolean-self
static_graphconstruction_nuts3.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='mean', 
                                                       graphname = 'geographical_neighbors1')
#   boolean-nonself
static_graphconstruction_nuts3.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='0', 
                                                       graphname = 'geographical_neighbors2')
#   numerical-self
static_graphconstruction_nuts3.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection = 'mean', 
                                                       normalization='rowwise', 
                                                       graphname = 'geographical_neighbors3')
#   numerical-nonself
static_graphconstruction_nuts3.generate_graphstructure(method='geographic_neighbors', 
                                                       self_connection='0', 
                                                       normalization='rowwise', 
                                                       graphname = 'geographical_neighbors4')

#   sparse - long distance gravity model
static_graphconstruction_nuts3.generate_graphstructure(method             = 'gravity_model', 
                                                 self_connection    = '0',
                                                 max_distance       = 1000_000,
                                                 top_k              = 3,
                                                 alpha              = 1,
                                                 decay              = 1,
                                                 normalization     = 'rowwise',
                                                 graphname = 'gravity1'
                                                 )
#   dense - long distance gravity model
static_graphconstruction_nuts3.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 10,
                                          alpha              = 1,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity2'
                                          )
#   sparse - short distance gravity model
static_graphconstruction_nuts3.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 3,
                                          alpha              = 2,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity3'
                                          )
#   dense - short distance gravity model
static_graphconstruction_nuts3.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 10,
                                          alpha              = 2,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity4'
                                          )
#   medium-dense - medium-distance gravity model
static_graphconstruction_nuts3.generate_graphstructure(method             = 'gravity_model', 
                                          self_connection    = '0',
                                          max_distance       = 1000_000,
                                          top_k              = 7,
                                          alpha              = 1.5,
                                          decay              = 1,
                                          normalization     = 'rowwise',
                                          graphname = 'gravity5'
                                          )

# Commuter
#   static - 2024-1
#   low threshold
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 500,
    normalization      = 'rowwise',
    graphname           = 'static_commuter24_1',
    year                = '2024'
)
#   static - 2024-2
#   medium threshold
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 1000,
    normalization      = 'rowwise',
    graphname           = 'static_commuter24_2',
    year                = '2024'
)
#   static - 2024-3
#   high threshold
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 2500,
    normalization      = 'rowwise',
    graphname           = 'static_commuter24_3',
    year                = '2024'
)
#   static - 2024-4
#   top_k=4
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 0,
    normalization      = 'rowwise',
    graphname           = 'static_commuter24_4',
    year                = '2024',
    top_k               = 4
)

# Commuter
#   static - 2014-1
#   low threshold
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 500,
    normalization      = 'rowwise',
    graphname           = 'static_commuter14_1',
    year                = '2014'
)
#   static - 2014-2
#   medium threshold
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 1000,
    normalization      = 'rowwise',
    graphname           = 'static_commuter14_2',
    year                = '2014'
)
#   static - 2014-3
#   high threshold
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 2500,
    normalization      = 'rowwise',
    graphname           = 'static_commuter14_3',
    year                = '2014'
)
#   static - 2014-4
#   top_k=4
static_graphconstruction_nuts3.generate_graphstructure(
    method              = 'commuter', 
    self_connection     = 'eps',
    commuting_threshold = 0,
    normalization      = 'rowwise',
    graphname           = 'static_commuter14_4',
    year                = '2014',
    top_k               = 4
)


# static_graphconstruction_nuts3.graph_registry.save_graphentry('ALL')

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


normalization method is None
    ✓ Graph registered  : identity_graph successfully registered
normalization method is None
    ✓ Graph registered  : geographical_neighbors1 successfully registered
normalization method is None
    ✓ Graph registered  : geographical_neighbors2 successfully registered
    ✓ Graph registered  : geographical_neighbors3 successfully registered
    ✓ Graph registered  : geographical_neighbors4 successfully registered
    ✓ Graph registered  : gravity1 successfully registered
    ✓ Graph registered  : gravity2 successfully registered
    ✓ Graph registered  : gravity3 successfully registered
    ✓ Graph registered  : gravity4 successfully registered
    ✓ Graph registered  : gravity5 successfully registered
✓ commuting-data loaded for ['2024']
    ✓ Graph registered  : static_commuter24_1 successfully registered
✓ commuting-data loaded for ['2024']
    ✓ Graph registered  : static_commuter24_2 successfully registered
✓ commuting-data loaded for ['2024']
    ✓ 

In [23]:
# Create a test graph
static_graphconstruction_nuts3.generate_graphstructure(
    method='commuter', 
    self_connection='mean',        # ✅ Strong self-loops
    commuting_threshold=1000,
    normalization='symmetric',    # ✅ Better for GNNs  
    graphname='commuter_symmetric_test',
    year='2024'
)
# Check it:
analyze_graph_structure(
    static_graphconstruction_nuts3.graph_registry.get_entry('commuter_symmetric_test').structure,
    'commuter_symmetric_test'
)

# static_graphconstruction_nuts3.graph_registry.save_graphentry('commuter_symmetric_test')

✓ commuting-data loaded for ['2024']
    ⚠️ warning          : commuter_symmetric_test already exists, please rename the already existing entry. New entry wasn't registered

commuter_symmetric_test self-loop analysis:
  Self-loops: 400/400
  Self-loop weight mean: 0.796063
  Self-loop weight std: 0.123647
  Self-loop weight min/max: 0.337367 / 0.983806
  Non-self edges: 2285
  Non-self weight mean: 0.034966
  Non-self weight std: 0.042860
  Self/Non-self ratio: 22.7670
  Node 0: 92.54% self-weight
  Node 1: 80.93% self-weight
  Node 2: 83.34% self-weight
  Node 3: 91.75% self-weight
  Node 4: 92.09% self-weight


In [19]:
analyze_graph_structure(static_graphconstruction_nuts3.graph_registry.get_entry('static_commuter24_3').structure,'static_commuter24_3')


static_commuter24_3 self-loop analysis:
  Self-loops: 400/400
  Self-loop weight mean: 0.067500
  Self-loop weight std: 0.251200
  Self-loop weight min/max: 0.000000 / 1.000000
  Non-self edges: 1112
  Non-self weight mean: 0.335432
  Non-self weight std: 0.291173
  Self/Non-self ratio: 0.2012
  Node 0: 0.00% self-weight
  Node 1: 0.00% self-weight
  Node 2: 0.00% self-weight
  Node 3: 0.00% self-weight
  Node 4: 0.00% self-weight


In [11]:
from src.graphconstruction.containers import GraphStructure

import torch

# After normalization, before saving
def analyze_graph_structure(graph_structure: GraphStructure, graphname: str):
    edge_index = graph_structure.edge_index
    edge_weight = graph_structure.edge_weight
    
    # Check self-loops
    self_loop_mask = edge_index[0] == edge_index[1]
    num_self_loops = self_loop_mask.sum().item()
    
    if num_self_loops > 0:
        self_loop_weights = edge_weight[self_loop_mask]
        print(f"\n{graphname} self-loop analysis:")
        print(f"  Self-loops: {num_self_loops}/{graph_structure.num_nodes}")
        print(f"  Self-loop weight mean: {self_loop_weights.mean():.6f}")
        print(f"  Self-loop weight std: {self_loop_weights.std():.6f}")
        print(f"  Self-loop weight min/max: {self_loop_weights.min():.6f} / {self_loop_weights.max():.6f}")
    
    # Check non-self edges
    non_self_mask = ~self_loop_mask
    if non_self_mask.sum() > 0:
        non_self_weights = edge_weight[non_self_mask]
        print(f"  Non-self edges: {non_self_mask.sum().item()}")
        print(f"  Non-self weight mean: {non_self_weights.mean():.6f}")
        print(f"  Non-self weight std: {non_self_weights.std():.6f}")
        
        # Ratio of self to non-self
        if num_self_loops > 0:
            ratio = self_loop_weights.mean() / (non_self_weights.mean() + 1e-10)
            print(f"  Self/Non-self ratio: {ratio:.4f}")
    
    # Check if effectively identity
    total_weight_per_node = torch.zeros(graph_structure.num_nodes)
    for i in range(graph_structure.num_nodes):
        mask = edge_index[0] == i
        total_weight_per_node[i] = edge_weight[mask].sum()
        
        self_weight = edge_weight[(edge_index[0] == i) & (edge_index[1] == i)].sum()
        if total_weight_per_node[i] > 0:
            self_proportion = self_weight / total_weight_per_node[i]
            if i < 5:  # Print first 5 nodes
                print(f"  Node {i}: {self_proportion:.2%} self-weight")


In [ ]:
# dynamic_graphconstruction = DynamicGraphOrchestrator(data_orchestrator)

# # dynamic_graphconstruction.generate_graphstructure(method = 'gravity_model', time_window=['2001-01-01','2025-01-01'], frequency = 'yearly', graphname='dynamic_gravity', normalization='rowwise')

# # Dynamic gravity graphs
# # number 1: sparse - long distance gravity model
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'gravity_model', 
#     time_window         = ['2005-01-01','2024-01-01'],
#     frequency           = 'yearly',    
#     self_connection     = '0',
#     max_distance        = 1000_000,
#     top_k               = 3,
#     alpha               = 1,
#     decay               = 1,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_gravity1'
# )

# # number 2: dense - long distance gravity model
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'gravity_model', 
#     time_window         = ['2005-01-01','2024-01-01'],
#     frequency           = 'yearly',    
#     self_connection     = '0',
#     max_distance        = 1000_000,
#     top_k               = 10,
#     alpha               = 1,
#     decay               = 1,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_gravity2'
# )

# # number 3: sparse - short distance gravity model
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'gravity_model', 
#     time_window         = ['2005-01-01','2024-01-01'],
#     frequency           = 'yearly',    
#     self_connection     = '0',
#     max_distance        = 1000_000,
#     top_k               = 3,
#     alpha               = 2,
#     decay               = 1,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_gravity3'
# )

# # number 4: dense - short distance gravity model
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'gravity_model', 
#     time_window         = ['2005-01-01','2024-01-01'],
#     frequency           = 'yearly',    
#     self_connection     = '0',
#     max_distance        = 1000_000,
#     top_k               = 10,
#     alpha               = 2,
#     decay               = 1,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_gravity4'
# )

# # number 5: medium-dense - medium-distance gravity model
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'gravity_model', 
#     time_window         = ['2005-01-01','2024-01-01'],
#     frequency           = 'yearly',    
#     self_connection     = '0',
#     max_distance        = 1000_000,
#     top_k               = 7,
#     alpha               = 1.5,
#     decay               = 1,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_gravity5'
# )

# # Dynamic commuter graphs
# #   number 1: low threshold
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'commuter', 
#     time_window         = ['2002-01-01','2024-01-01'],
#     frequency           = 'yearly',    
#     self_connection     = 'eps',
#     commuting_threshold = 500,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_commuter1'
# )
# #   number 2: medium threshold
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'commuter', 
#     time_window         = ['2002-01-01','2024-01-01'],  
#     frequency           = 'yearly',      
#     self_connection     = 'eps',
#     commuting_threshold = 1000,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_commuter2'
# )
# #   number 3: high threshold
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'commuter', 
#     time_window         = ['2002-01-01','2024-01-01'],    
#     frequency           = 'yearly',    
#     self_connection     = 'eps',
#     commuting_threshold = 2500,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_commuter3'
# )
# #   number 4: top-k
# dynamic_graphconstruction.generate_graphstructure(
#     method              = 'commuter', 
#     time_window         = ['2002-01-01','2024-01-01'],    
#     frequency           = 'yearly',    
#     self_connection     = 'eps',
#     commuting_threshold = 0,
#     normalization       = 'rowwise',
#     graphname           = 'dynamic_commuter4',
#     top_k               = 4
# )
# # dynamic_graphconstruction.graph_registry.save_graphentry('ALL')

generating graph => dynamic_gravity1: 100%|███████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.61it/s]


    ✓ Graph registered  : dynamic_gravity1 successfully registered


generating graph => dynamic_gravity2: 100%|███████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.40it/s]


    ✓ Graph registered  : dynamic_gravity2 successfully registered


generating graph => dynamic_gravity3: 100%|███████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.62it/s]


...

    ✓ Graph registered  : dynamic_commuter2 successfully registered
✓ commuting-data loaded for ['2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


generating graph => dynamic_commuter3: 100%|██████████████████████████████████████████████████| 23/23 [00:00<00:00, 25.97it/s]


    ✓ Graph registered  : dynamic_commuter3 successfully registered
✓ commuting-data loaded for ['2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


generating graph => dynamic_commuter4: 100%|██████████████████████████████████████████████████| 23/23 [00:01<00:00, 12.94it/s]

    ✓ Graph registered  : dynamic_commuter4 successfully registered
